# Phase 10 — Informatique décisionnelle

## Pourquoi cette phase est importante

Les phases 5 à 9 publient des fichiers et des pages HTML. Une équipe d'investment analytics s'en sert ensuite dans un outil où l'on filtre un livre, puis où les graphiques du même écran se limitent à ce livre. Power BI est cet outil. Les trois livres sont hypothétiques. Le fonds iShares est une série réelle de rendement, pas un quatrième livre de poids.

Le Sharpe, la VaR, l'expected shortfall et les chocs lisent la cellule publiée. Deux mesures recalculent en DAX, avec la définition du pipeline, uniquement pour la réconciliation Excel : `Rendement chaîné calculé` et `Drawdown maximal calculé`. Elles ne sont pas posées sur une page. La page Performance calcule aussi, à partir des rendements mensuels publiés, le multiple contre le SARON, le multiple contre le fonds et le rendement glissant sur 12 mois. Le fonds y figure : sa ligne est dans `active_return.csv`, avec un écart nul contre lui-même. Les pages d'allocation, de suivi et de chocs ne proposent pas le fonds dans le filtre, parce qu'elles n'ont pas de poids ni de choc pour cette série.

## Ce que contient le projet

Le fichier à ouvrir est `dashboards/powerbi/Investment Analytics.pbip`, dans Power BI Desktop. Le modèle lit les fichiers publiés au moment du rafraîchissement. Le paramètre `ProjectRoot` est le dossier du projet. Le paramètre `SeuilPaires` est le seuil de lecture des paires, le même `0,5` que la phase 8. Ce sont des paramètres, pas des résultats mesurés.

Six pages. Les graphiques sont les mêmes séries que les pages HTML des phases 6 à 9. Power BI ajoute le filtre par livre.

| Page | Question | Fichiers lus |
|---|---|---|
| Portfolio Overview | Où en est le livre au dernier mois publié ? | `monthly_report`, allocations par secteur, zone et devise, `limit_watch` |
| Suivi | Comment évoluent l'effectif, les plafonds, la rotation, les contributions et les dix premières lignes ? | `concentration_history`, `limit_watch`, `turnover`, `sector_contribution`, `allocation_by_holding` |
| Performance Analytics | Quelle performance cumulée, quel drawdown, quel écart au fonds et au SARON, quel rendement glissant ? | `portfolio_path`, `performance_metrics`, `active_return` |
| Risk Analytics | Quelle volatilité, quelle queue, quelle concentration, quelles dépendances ? | `risk_by_strategy`, `var_tail`, `monthly_report`, `concentration_history`, `universe_correlation`, `correlation_matrix` |
| Stress Testing | Que devient le chemin si l'on applique un choc déjà écrit ? | `stress_results`, `stress_paths` |
| Data Quality | Le dernier contrôle et le registre ODD sont-ils lisibles ? | `audit/data_control_center.xlsx`, `odd_assessment.csv` |

Deux groupes de filtres. Portfolio Overview, Suivi et Stress Testing partagent le groupe « portefeuille » : le fonds n'y est pas proposé. Performance Analytics et Risk Analytics partagent le groupe « livre », qui inclut le fonds, parce que ces pages lisent une série de rendement. Data Quality n'a pas de filtre de livre : le contrôle et le statut ODD portent sur les jeux et sur le registre.

Le graphique des contributions est découpé en trois panneaux, un par livre. Chaque panneau lit un seul livre. Le fonds n'y figure pas : la feuille de contributions sectorielles ne contient que les trois livres.

La page Risque ajoute la concentration du mois du rapport : titres effectifs, indice de Herfindahl, multiple, puis la série des titres effectifs. Ces visuels lisent les trois livres. Le fonds n'y est pas, ces fichiers ne le contenant pas. La matrice de Pearson publiée est remise en tableau, hors diagonale, ordonnée par le lien le plus fort. Le tableau des paires, plus bas, ne garde une paire que si sa corrélation atteint `SeuilPaires`, et une seule fois. Le chiffre affiché reste celui du fichier.

Sur Stress Testing, le graphique du temps est l'écart entre le chemin choqué et le chemin sans choc, pour le marché, la récession et la volatilité. Sans choc est la ligne zéro et n'est pas tracé. Le choc de taux ne change pas le rendement du portefeuille, donc il n'est pas sur ce graphique. Il reste dans les barres, parce qu'il change le Sharpe. Le marché ne s'écarte qu'au dernier mois. La récession ne s'écarte que sur ses derniers mois.

## Qui rafraîchit les chiffres

`python -m src.pipeline.refresh` réécrit les CSV et le classeur de qualité. Il n'ouvre pas Power BI. Dans Desktop, Actualiser relit ces fichiers. Les mesures sont des formules DAX : elles prennent la valeur de la colonne dans le filtre du visuel. Aucun résultat n'est inscrit dans la page.

La cellule suivante vérifie que les six pages existent et que chaque visuel pointe vers une colonne ou une mesure du modèle.


In [1]:

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

POWERBI = PROJECT_ROOT / "dashboards" / "powerbi"
REPORT = POWERBI / "Investment Analytics.Report"
MODEL = POWERBI / "Investment Analytics.SemanticModel"
pages = json.loads((REPORT / "definition" / "pages" / "pages.json").read_text(encoding="utf-8"))
expected = ["PortfolioOverview", "Monitoring", "PerformanceAnalytics", "RiskAnalytics", "StressTesting", "DataQuality"]
if pages["pageOrder"] != expected:
    raise RuntimeError("The dashboard pages are not in the written order.")
fields = set()
for path in (MODEL / "definition" / "tables").glob("*.tmdl"):
    table = path.stem
    for line in path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped.startswith("column "):
            fields.add((table, stripped.removeprefix("column ").strip("'")))
        if stripped.startswith("measure "):
            fields.add((table, stripped.split("=", 1)[0].removeprefix("measure ").strip().strip("'").replace("''", "'")))
visuals = list((REPORT / "definition" / "pages").glob("*/visuals/*/visual.json"))
if len(visuals) < 20:
    raise RuntimeError("The report has too few visuals for the dashboards.")
for path in visuals:
    payload = json.loads(path.read_text(encoding="utf-8"))
    query = payload.get("visual", {}).get("query", {}).get("queryState", {})
    for role in query.values():
        for projection in role.get("projections", []):
            holder = projection["field"]
            kind = "Column" if "Column" in holder else "Measure"
            ref = holder[kind]
            pair = (ref["Expression"]["SourceRef"]["Entity"], ref["Property"])
            if pair not in fields:
                raise RuntimeError(f"{path.name} asks for {pair}, which is not in the model.")
expressions = (MODEL / "definition" / "expressions.tmdl").read_text(encoding="utf-8")
if "SeuilPaires" not in expressions or "ProjectRoot" not in expressions:
    raise RuntimeError("The model has no path parameter or no pair threshold.")
root = expressions.split('"', 2)[1]
if not (Path(root) / "data" / "08_stress" / "stress_results.csv").is_file():
    raise RuntimeError("The Power BI path parameter does not reach the published stress file.")
print("Pages : " + ", ".join(pages["pageOrder"]))
print(f"Visuels : {len(visuals)}. Champs du modèle : {len(fields)}.")
print("Le projet est prêt à être ouvert dans Power BI Desktop.")


Pages : PortfolioOverview, Monitoring, PerformanceAnalytics, RiskAnalytics, StressTesting, DataQuality
Visuels : 49. Champs du modèle : 204.
Le projet est prêt à être ouvert dans Power BI Desktop.


## Dictionnaire

**Livre.** Un des trois portefeuilles de la phase 5. Le filtre porte ce nom.

**Choc.** Un des quatre chocs de la phase 9, plus la ligne sans choc.

**Seuil de lecture des paires.** Paramètre du modèle. Il reprend le seuil de la phase 8. Il ne dit pas qu'une paire est interdite.

**Actualiser.** Commande de Power BI Desktop qui relit les fichiers. Elle ne réestime pas les poids.
